In [26]:
from os.path import join as join_path
from tqdm.notebook import tqdm
import sys
import pandas as pd
import numpy as np
from sklearn.metrics import adjusted_mutual_info_score as ami
from sklearn.metrics import adjusted_rand_score as ari

sys.path.append("/export/share/peters57dm/Verbund/deepsync/")
from helper.deep import relabel_negative_ones
from helper.utils import load_json_as_dict
from helper.datasets import (
    load_example,
    load_usps,
    load_htru,
    load_pendigits,
    load_optdigits,
    load_mnist,
    load_letterrecognition,
    load_coil20,
    load_coil100,
    load_har,
    load_mice,
    load_weizmann,
    load_cifar10,
    load_cifar100,
    load_fmnist,
    load_data,
)


# Comparison 406: DeepSync predicts the true K

In [ ]:
results_path = "/export/share/peters57dm/Verbund/deepsync/results/experiments/comparison406/ae_sync_loss/knn_label_assignment"
datasets_loaders = [
    load_example,
    load_usps,
    load_htru,
    load_pendigits,
    load_optdigits,
    load_mnist,
    load_letterrecognition,
    load_coil20,
    load_coil100,
    load_har,
    load_mice,
    load_weizmann,
    load_fmnist,
]
N_MODELS = 5


In [28]:
results = {}
for ds_loader in tqdm(datasets_loaders, total=len(datasets_loaders)):
    data, gt_labels, data_name = load_data(ds_loader)
    gt_labels = gt_labels.numpy()

    _aris = []
    _amis = []
    for i in range(N_MODELS):
        _tracker_path = join_path(results_path, data_name, f"model_0{i}", "trackers", "eval_tracker.json")
        _r = load_json_as_dict(_tracker_path)
        _preds = np.array(_r["predicted_labels"])
        _preds = relabel_negative_ones(_preds)
        _ari = ari(_preds, gt_labels)
        _ami = ami(_preds, gt_labels)
        _aris.append(_ari)
        _amis.append(_ami)
    results[data_name] = {'ARI' : f"{np.mean(_aris):.4f}±{np.std(_aris):.4f}",
                        'AMI' : f"{np.mean(_amis):.4f}±{np.std(_amis):.4f}"}

  0%|          | 0/13 [00:00<?, ?it/s]

In [29]:
pd.DataFrame(results)

,example,USPS,htru,pendigits,optdigits,MNIST,letterrecognition,coil20,coil100,HAR,mice,weizmann,FMNIST
ARI,0.9410±0.0182,0.7239±0.0402,0.5918±0.0244,0.7246±0.0065,0.8566±0.0252,0.7395±0.0196,0.1831±0.0208,0.4896±0.0258,0.5285±0.0233,0.4564±0.0477,0.2183±0.0232,0.2977±0.0178,0.3897±0.0206
AMI,0.8736±0.0369,0.7408±0.0283,0.2597±0.0152,0.6833±0.0257,0.8636±0.0156,0.7965±0.0062,0.5118±0.0111,0.5456±0.0321,0.6093±0.0162,0.5942±0.0239,0.3284±0.0154,0.5525±0.0167,0.5734±0.0069


In [ ]:
pd.DataFrame(results).to_excel("/export/share/peters57dm/Verbund/deepsync/results/experiments/comparison406/ari_ami_with_distinct_noise_labels.xlsx")

# Comparison 404 - 405: true K was provided
there is an issue with weizmann results here.

In [31]:
results_path_1 = "/export/share/peters57dm/Verbund/deepsync/results/experiments/comparison404/ae_sync_loss/knn_label_assignment"
results_path_2 = "/export/share/peters57dm/Verbund/deepsync/results/experiments/comparison405/ae_sync_loss/knn_label_assignment"

results = {}
for ds_loader in tqdm(datasets_loaders, total=len(datasets_loaders)):
    data, gt_labels, data_name = load_data(ds_loader)
    gt_labels = gt_labels.numpy()

    _aris = []
    _amis = []
    for i in range(N_MODELS):
        try :
            _tracker_path = join_path(results_path_1, data_name, f"model_0{i}", "trackers", "eval_tracker.json")
            _r = load_json_as_dict(_tracker_path)
        except FileNotFoundError:
            _tracker_path = join_path(results_path_2, data_name, f"model_0{i}", "trackers", "eval_tracker.json")
            _r = load_json_as_dict(_tracker_path)
        _preds = np.array(_r["predicted_labels"])
        _preds = relabel_negative_ones(_preds)
        _ari = ari(_preds, gt_labels)
        _ami = ami(_preds, gt_labels)
        _aris.append(_ari)
        _amis.append(_ami)
    results[data_name] = {'ARI' : f"{np.mean(_aris):.4f}±{np.std(_aris):.4f}",
                        'AMI' : f"{np.mean(_amis):.4f}±{np.std(_amis):.4f}"}

  0%|          | 0/12 [00:00<?, ?it/s]

In [32]:
pd.DataFrame(results)

,example,USPS,htru,pendigits,optdigits,MNIST,letterrecognition,coil20,coil100,HAR,mice,FMNIST
ARI,0.9396±0.0166,0.7218±0.0251,0.5891±0.0852,0.7040±0.0099,0.7939±0.0627,0.7432±0.0487,0.0889±0.0200,0.4820±0.0209,0.5276±0.0178,0.4421±0.0568,0.2168±0.0255,0.3896±0.0257
AMI,0.8709±0.0325,0.7096±0.0246,0.2441±0.0446,0.6664±0.0244,0.8468±0.0219,0.8070±0.0210,0.4314±0.0111,0.5327±0.0233,0.5905±0.0174,0.6112±0.0360,0.3237±0.0173,0.5850±0.0256


In [33]:
pd.DataFrame(results).to_excel("/export/share/peters57dm/Verbund/deepsync/results/experiments/comparison404/ari_ami_with_distinct_noise_labels.xlsx")